In [4]:

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

# Load dataset
dataset = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

numeric_cols = [
    'URLLength',
    'DomainLength',
    'TLDLength',
    'NoOfSubDomain',
    'CharContinuationRate',
    'LetterRatioInURL',
    'DegitRatioInURL',
    'NoOfLettersInURL',
    'NoOfDegitsInURL',
    'SpacialCharRatioInURL'
]

dominant_features = [
    'IsHTTPS',
    'NoOfOtherSpecialCharsInURL'
]

near_zero_features = [
    'NoOfQMarkInURL',
    'NoOfEqualsInURL',
    'NoOfAmpersandInURL',
    'IsDomainIP'
]

numeric_cols = [c for c in numeric_cols if c not in dominant_features]
numeric_cols = [c for c in numeric_cols if c not in near_zero_features]
numeric_cols = [c for c in numeric_cols if c in dataset.columns]
print("Features used:", numeric_cols)

# Check for possible leakage
print("Duplicate URLs:", dataset["URL"].duplicated().sum())
conflicting = dataset.groupby("URL")["label"].nunique()
print("URLs with conflicting labels:", (conflicting > 1).sum())

# Prepare data
X = dataset[numeric_cols]
y = dataset["label"]
groups = dataset["Domain"]

# Build pipeline
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=150, criterion='entropy', 
    random_state=42, n_jobs=-1, max_depth=10))
])

# Domain-isolated train/test split
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups))
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# Baseline
baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print("\n--- Baseline ---")
print("Accuracy: {:.2f}%".format(
    accuracy_score(y_test, baseline_pred) * 100
))

# Test results
print("\n--- Random Forest Results (reduced features) ---")
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nAccuracy: {:.2f}%".format(
    accuracy_score(y_test, y_pred) * 100
))
print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Phishing", "Legitimate"]
))

# Group cross validation
cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring="accuracy"
)
print("\n--- GroupKFold Cross Validation ---")
print("Mean Accuracy: {:.2f}%".format(cv_scores.mean() * 100))
print("Std Dev: {:.2f}%".format(cv_scores.std() * 100))

# Feature importance
pipeline.fit(X, y)
importance = pd.Series(
    pipeline.named_steps["classifier"].feature_importances_,
    index=numeric_cols
).sort_values(ascending=False)
print("\n--- Feature Importance ---")
print(importance)

Features used: ['URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain', 'CharContinuationRate', 'LetterRatioInURL', 'DegitRatioInURL', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'SpacialCharRatioInURL']
Duplicate URLs: 425
URLs with conflicting labels: 0

--- Baseline ---
Accuracy: 57.34%

--- Random Forest Results (reduced features) ---

Confusion Matrix
[[17275  2841]
 [  143 26900]]

Accuracy: 93.67%

Classification Report
              precision    recall  f1-score   support

    Phishing       0.99      0.86      0.92     20116
  Legitimate       0.90      0.99      0.95     27043

    accuracy                           0.94     47159
   macro avg       0.95      0.93      0.93     47159
weighted avg       0.94      0.94      0.94     47159


--- GroupKFold Cross Validation ---
Mean Accuracy: 93.71%
Std Dev: 0.36%

--- Feature Importance ---
LetterRatioInURL         0.195300
SpacialCharRatioInURL    0.157593
NoOfDegitsInURL          0.140795
DegitRatioInURL          0.140320
URLLen

Random Forest matches XGBoost's accuracy (~99.7%) when allowed to grow deep, unrestricted trees, but that depth makes the program run much slower. Capping max_depth to speed it up caused accuracy to drop sharply (to ~93%), suggesting the decision boundary for these features needs substantial nested splitting. XGBoost achieves the same accuracy with shallower, boosted trees, making it the more efficient choice here.